<a class="anchor" id="2">

# Notebook 5: Streaming
</a>

### Group 16 Members:

- Ana Margarida Macedo (20250405)
- Catarina Aboim (20250375)
- Margarida Craveiro (20250346)
- Matilde Simões (20250382)
- Lourenço Silva (20250453)

<a class="anchor" id="1">

## **Structured Streaming: Real-Time NYC Yellow Taxi Analytics**
</a>

In this notebook, we apply **Spark Structured Streaming** to the NYC Yellow Taxi dataset, using **Apache Kafka** as both the input source and output sink, exactly as in a production real-time pipeline.

The pipeline follows the 5-step recipe from the lectures:

```
readStream (Kafka) → parse → withWatermark + window + groupBy → writeStream (Kafka) + checkpoint
```

<a class="anchor" id="1">

# **1. Overview & Architecture**
</a>

Before starting this approach is important to give a reminder a small context to remember the dataset chosen for this part:

The data is the **NYC Yellow Taxi Trip Records** cleaned and saved in Notebook 1 (`data/clean/taxi_clean.parquet`). Each record represents one completed trip.

| Column | Description |
|---|---|
| `tpep_pickup_datetime` | Trip start — used as **event time** |
| `tpep_dropoff_datetime` | Trip end |
| `PULocationID` | Pickup zone — used for demand aggregations |
| `VendorID` | Vendor identifier |
| `fare_amount` | Base fare (USD) |
| `tip_amount` | Tip (USD) |
| `trip_distance` | Distance (miles) |
| `total_amount` | Total charge (USD) |

Because no live taxi GPS feed is available, we **simulate** one by reading the Parquet file in a background thread and **producing** each trip as a JSON message into the `taxi-trips` Kafka topic exactly as a real taxi meter would push events.

A separate producer script (`taxi_producer.py`) reads the cleaned taxi Parquet file and publishes events to two Kafka topics:

| Topic | Cadence | Payload |
|---|---|---|
| `trips` | every ~5 s (50 events/cycle) | `pickup_ts`, `dropoff_ts`, `PULocationID`, `DOLocationID`, `passenger_count`, `trip_distance`, `fare_amount`, `tip_amount`, `total_amount`, `payment_type` |
| `long_trips` | subset — only trips > 10 miles | same payload, filtered |

We use them to demonstrate the standard streaming patterns:

1. Reading a Kafka topic as a typed streaming DataFrame
2. Sinks and output modes (`append`, `update`, `complete`)
3. Windowed aggregations with watermarks
4. Stream-stream join: correlate `trips` with `long_trips` to surface high-fare long-haul events

Structured Streaming treats a stream as a table that updates over time, so the same DataFrame API works on both bounded and unbounded data.

<a class="anchor" id="2">

## **1.2. Dataset Context**
</a>

The data used in this notebook is the **NYC Yellow Taxi Trip Records**, previously cleaned and consolidated in Notebook 1. It covers four months 
of trips (January 2015 and January–March 2016) and is stored locally at `data/clean/taxi_clean.parquet`.

Each record represents a single completed trip and contains the following key attributes for streaming analysis:

| Column | Description |
|---|---|
| `tpep_pickup_datetime` | Trip start timestamp — used as **event time** |
| `tpep_dropoff_datetime` | Trip end timestamp |
| `PULocationID` | Pickup zone — used for demand aggregations |
| `VendorID` | Vendor identifier — used for revenue monitoring |
| `fare_amount` | Base fare in USD |
| `tip_amount` | Tip in USD |
| `trip_distance` | Distance in miles |
| `total_amount` | Total charge in USD |

<a class="anchor" id="2">

# **2. Environment Setup**
</a>

In [ ]:
import pyspark.sql.functions as F
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, DoubleType, IntegerType,
)

KAFKA_BOOTSTRAP_SERVERS = "localhost:8098"

In [2]:
spark = (
    SparkSession.builder
    .appName("kafka_streaming")
    .config("spark.streaming.stopGracefullyOnShutdown", True)
    .config("spark.jars.packages", "org.apache.spark:spark-sql-kafka-0-10_2.12:3.5.0")
    .config("spark.sql.shuffle.partitions", 4)
    .master("local[*]")
    .getOrCreate()
)

:: loading settings :: url = jar:file:/system/conda/miniconda3/envs/cloudspace/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /home/zeus/.ivy2/cache
The jars for the packages stored in: /home/zeus/.ivy2/jars
org.apache.spark#spark-sql-kafka-0-10_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-43a2d23c-41ee-43b1-90d9-b0644607a902;1.0
	confs: [default]
	found org.apache.spark#spark-sql-kafka-0-10_2.12;3.5.0 in central
	found org.apache.spark#spark-token-provider-kafka-0-10_2.12;3.5.0 in central
	found org.apache.kafka#kafka-clients;3.4.1 in central
	found org.lz4#lz4-java;1.8.0 in central
	found org.xerial.snappy#snappy-java;1.1.10.3 in central
	found org.slf4j#slf4j-api;2.0.7 in central
	found org.apache.hadoop#hadoop-client-runtime;3.3.4 in central
	found org.apache.hadoop#hadoop-client-api;3.3.4 in central
	found commons-logging#commons-logging;1.1.3 in central
	found com.google.code.findbugs#jsr305;3.0.0 in central
	found org.apache.commons#commons-pool2;2.11.1 in central
:: resolution report :: resolve 378ms :: artifacts dl 13ms
	:: mod

<a class="anchor" id="2">

# **2. Reading the trips stream**

(transform the database)
</a>

We subscribe to the **`trips`** topic, declare the JSON schema, and parse the Kafka **`value`** column into a typed DataFrame. The result, **`trips_df`**, is the canonical streaming DataFrame we´ll reuse throughout the rest the notebook.

In [3]:
# Define the schema for the incoming trip data
trip_schema = StructType([
    StructField("tpep_pickup_datetime",  StringType(),  True),
    StructField("tpep_dropoff_datetime", StringType(),  True),
    StructField("PULocationID",          IntegerType(), True),
    StructField("DOLocationID",          IntegerType(), True),
    StructField("passenger_count",       IntegerType(), True),
    StructField("trip_distance",         DoubleType(),  True),
    StructField("fare_amount",           DoubleType(),  True),
    StructField("tip_amount",            DoubleType(),  True),
    StructField("total_amount",          DoubleType(),  True),
    StructField("payment_type",          IntegerType(), True),
])

# Read the raw trip data from the Kafka topic "trips"
raw_trips = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "trips")
    .option("startingOffsets", "earliest")
    .load()
)

# Parse the JSON value into a typed DataFrame and perform basic cleaning
trips_df = (
    raw_trips
    .select(F.from_json(F.col("value").cast("string"), trip_schema).alias("v"))
    .select(
        F.to_timestamp(F.col("v.tpep_pickup_datetime"),  "yyyy-MM-dd HH:mm:ss").alias("pickup_ts"),
        F.to_timestamp(F.col("v.tpep_dropoff_datetime"), "yyyy-MM-dd HH:mm:ss").alias("dropoff_ts"),
        F.col("v.PULocationID")    .alias("PULocationID"),
        F.col("v.DOLocationID")    .alias("DOLocationID"),
        F.col("v.passenger_count") .alias("passenger_count"),
        F.col("v.trip_distance")   .alias("trip_distance"),
        F.col("v.fare_amount")     .alias("fare_amount"),
        F.col("v.tip_amount")      .alias("tip_amount"),
        F.col("v.total_amount")    .alias("total_amount"),
        F.col("v.payment_type")    .alias("payment_type"),
    )
    .filter(F.col("pickup_ts").isNotNull())
    .filter(F.col("fare_amount") > 0)
    .filter(F.col("trip_distance") > 0)
)

Like the batch API, streaming operations are lazy - nothing happens until and call **`.start()`**.

<a class="anchor" id="2">

# **3. Sinks and output models**

(choose sink + output mode)
</a>

Structured Streaming supports several sinks (what deciding where we want our output to go (be stored)), we can have **Kafka**, **file** (parquet/JSON/CSV), foreach custom logic, **console** for debugging, and **memory** for interactive inspection from SQL. 

Each query also has an *output mode* that controls what gets written on each trigger:

- `append` — only new rows since the last trigger
- `update` — only rows whose aggregate value changed
- `complete` — the entire result table (only valid for aggregates)

For this project it was decided the **memory** sink with a query name, which lets us run ad-hoc Spark SQL against the rolling stream. For this case **append** is use as output mode because, one trip is an immutable event, once a trip ends it never changes, so we only over append new rows and never update existing ones.

In [4]:
trips_query = (
    trips_df.writeStream
    .outputMode("append")
    .queryName("trips_table")
    .format("memory")
    .start()
)

26/05/30 22:06:55 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-74a6b79a-4e9d-457d-a45d-47069d765e9a. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/30 22:06:55 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/30 22:06:55 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [ ]:
# Select done to ensure the streaming query is active
# and the in-memory table is being populated
spark.sql("""
    SELECT PULocationID,
           COUNT(*)                    AS trip_count,
           ROUND(AVG(fare_amount), 2)  AS avg_fare,
           ROUND(AVG(trip_distance),2) AS avg_distance,
           ROUND(SUM(total_amount), 2) AS total_revenue
    FROM trips_table
    GROUP BY PULocationID
    ORDER BY trip_count DESC
""").show()

+------------+----------+--------+------------+-------------+
|PULocationID|trip_count|avg_fare|avg_distance|total_revenue|
+------------+----------+--------+------------+-------------+
|         234|        94|    9.47|        1.81|      1146.64|
|         236|        90|   10.19|         2.2|      1164.07|
|         161|        80|   11.23|        2.42|      1143.61|
|         237|        78|    9.42|         1.7|       902.63|
|          48|        78|   12.04|        2.67|      1175.69|
|          79|        78|   10.38|        2.19|      1023.71|
|         230|        74|   10.13|        2.17|       931.67|
|          68|        70|   11.73|        2.78|      1054.21|
|         239|        68|    10.1|        2.19|        871.4|
|         162|        67|   11.57|        2.64|      1000.02|
|         249|        65|   10.49|        2.23|       848.26|
|         142|        65|    9.41|        1.84|       742.33|
|         107|        64|   11.13|        2.38|       904.21|
|       

<a class="anchor" id="6">

# **4. Windowed aggregations and watermarks**
</a>

Time-windowed aggregations allow to aggregate data over a sliding or tumbling time period rather than calculating them across the entire streaming. 

A **watermark** tells the engine how late an event can arrive and still be included in its window, this is what allows Spark to safely drop old window state from memory instead of keeping it forever. State for windows older than `max(event_time) − watermark` is dropped, which bounds the memory footprint of a long-running query.

Here we use a **3-minute tumbling window** on `pickup_ts` with a **30-second watermark** to compute per-location OHLC-style fare statistics.


In [ ]:
windowed = (
    trips_df
    .withWatermark("pickup_ts", "30 seconds")
    .groupBy(
        # Use a 3-minute tumbling window to capture short-term trends in pickup activity
        # and group by pickup location to identify hotspots
        F.window("pickup_ts", "3 minutes"),
        "PULocationID",
    )
    .agg(
        F.count("*")                       .alias("trip_count"),
        F.avg("fare_amount")               .alias("avg_fare"),
        F.min("fare_amount")               .alias("min_fare"),
        F.max("fare_amount")               .alias("max_fare"),
        F.sum("total_amount")              .alias("total_revenue"),
        F.avg("trip_distance")             .alias("avg_distance"),
    )
)

windowed_query = (
    windowed.writeStream
    .outputMode("append")
    .queryName("trips_aggregated")
    .format("memory")
    .start()
)

26/05/30 22:06:59 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-6c0ec27c-9f43-4e2a-b3c7-de98bf192b4e. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/30 22:06:59 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/30 22:06:59 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


In [12]:
spark.sql("SELECT * FROM trips_aggregated ORDER BY window").show(truncate=False)

+------------------------------------------+------------+----------+--------+--------+--------+-------------+------------+
|window                                    |PULocationID|trip_count|avg_fare|min_fare|max_fare|total_revenue|avg_distance|
+------------------------------------------+------------+----------+--------+--------+--------+-------------+------------+
|{2015-01-01 03:21:00, 2015-01-01 03:24:00}|229         |1         |6.0     |6.0     |6.0     |8.6          |0.95        |
|{2015-01-01 03:30:00, 2015-01-01 03:33:00}|211         |1         |6.0     |6.0     |6.0     |8.6          |1.08        |
|{2015-01-01 05:51:00, 2015-01-01 05:54:00}|79          |1         |16.0    |16.0    |16.0    |18.3         |4.18        |
|{2015-01-01 07:24:00, 2015-01-01 07:27:00}|48          |1         |4.5     |4.5     |4.5     |5.3          |0.82        |
|{2015-01-01 15:18:00, 2015-01-01 15:21:00}|107         |1         |7.5     |7.5     |7.5     |8.3          |1.24        |
|{2015-01-01 15:

<a class="anchor" id="6">

# **5. Stream-stream join**
</a>

We join the `trips` stream with the `long_trips` stream to answer:
*for each long-haul trip (> 10 miles), does a matching high-fare event (fare > $30) appear on the same pickup location within 2 minutes?*

This mirrors the class lab's news→trade-reaction join pattern:

Three design points worth flagging:

1. **Both sides must be watermarked.** Stream-stream joins require a watermark on each side so Spark can bound the join state and eventually emit results.
2. **Tight watermarks.** Since `pickup_ts` is the original trip timestamp (not wall clock), events from the producer arrive with some replay lag. We use `30 seconds` on both sides.
3. **Interval condition.** The join uses a `BETWEEN` on the pickup timestamp, making this a *time-bounded* join with finite state.

In [6]:
long_trip_schema = trip_schema  # identical schema, just a filtered topic

raw_long = (
    spark.readStream.format("kafka")
    .option("kafka.bootstrap.servers", KAFKA_BOOTSTRAP_SERVERS)
    .option("subscribe", "long_trips")
    .option("startingOffsets", "earliest")
    .load()
)

# Note: event_time is the original trip pickup timestamp from the parquet file,
# not the wall-clock time of arrival. Events replayed from historical data
# may arrive out of order, hence the watermark.
long_trips_df = (
    raw_long
    .select(F.from_json(F.col("value").cast("string"), long_trip_schema).alias("v"))
    .select(
        F.to_timestamp(F.col("v.tpep_pickup_datetime"), "yyyy-MM-dd HH:mm:ss").alias("pickup_ts"),
        F.col("v.PULocationID") .alias("PULocationID"),
        F.col("v.trip_distance").alias("trip_distance"),
        F.col("v.fare_amount")  .alias("fare_amount"),
        F.col("v.total_amount") .alias("total_amount"),
    )
    .filter(F.col("pickup_ts").isNotNull())
)

In [10]:
# Watermark each side. Since `event_time` is the producer's
# observation timestamp (always 'now') for both streams, events
# are never more than a few seconds late - so the watermarks can
# be tight. 
trips_wm     = trips_df.withWatermark("pickup_ts", "30 seconds").alias("t") # t stands for "trips"
long_trips_wm = long_trips_df.withWatermark("pickup_ts", "30 seconds").alias("l") # l stands for "long_trips"

# For each long trip, find any trip on the same pickup location
# with fare > $30 that fired within the next 2 minutes.
joined = trips_wm.join(
    long_trips_wm,
    F.expr("""
        t.PULocationID = l.PULocationID AND
        t.fare_amount > 30             AND
        t.pickup_ts BETWEEN l.pickup_ts AND l.pickup_ts + interval 2 minutes
    """),
)

# Roll up to one row per long trip — how many high-fare matches appeared?
reaction = (
    joined.groupBy("l.PULocationID", "l.pickup_ts", "l.trip_distance")
    .agg(
        F.count("*")             .alias("matching_high_fare_trips"),
        F.first("t.fare_amount") .alias("first_high_fare"),
        F.max("t.fare_amount")   .alias("max_fare_in_window"),
    )
)

reaction_query = (
    reaction.writeStream
    .outputMode("append")
    .queryName("long_trip_alerts")
    .format("memory")
    .start()
)

26/05/30 22:11:29 WARN ResolveWriteToStream: Temporary checkpoint location created which is deleted normally when the query didn't fail: /tmp/temporary-3e0f4594-052f-4cb9-914e-d5e0e5f6c169. If it's required to delete it under any circumstances, please set spark.sql.streaming.forceDeleteTempCheckpointLocation to true. Important to know deleting temp checkpoint folder is best effort.
26/05/30 22:11:29 WARN ResolveWriteToStream: spark.sql.adaptive.enabled is not supported in streaming DataFrames/Datasets and will be disabled.


26/05/30 22:11:29 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.
26/05/30 22:11:29 WARN AdminClientConfig: These configurations '[key.deserializer, value.deserializer, enable.auto.commit, max.poll.records, auto.offset.reset]' were supplied but are not used yet.


**Expect the first rows to appear ~3–4 minutes after starting the producer.** Stream-stream joins only support `append` output mode, meaning Spark holds each window's output until the watermark has passed the window's end.

In [13]:
spark.sql("""
    SELECT PULocationID,
           pickup_ts,
           trip_distance,
           matching_high_fare_trips,
           ROUND(first_high_fare, 2)     AS first_high_fare,
           ROUND(max_fare_in_window, 2)  AS max_fare_in_window
    FROM long_trip_alerts
    ORDER BY pickup_ts DESC
""").show(truncate=False)

+------------+-------------------+-------------+------------------------+---------------+------------------+
|PULocationID|pickup_ts          |trip_distance|matching_high_fare_trips|first_high_fare|max_fare_in_window|
+------------+-------------------+-------------+------------------------+---------------+------------------+
|138         |2016-03-29 09:42:50|11.81        |1                       |40.0           |40.0              |
|132         |2016-03-28 16:43:24|17.8         |1                       |52.0           |52.0              |
|132         |2016-03-28 08:53:09|28.46        |1                       |52.0           |52.0              |
|97          |2016-03-27 23:29:00|13.64        |1                       |44.5           |44.5              |
|164         |2016-03-27 21:14:41|10.6         |1                       |34.0           |34.0              |
|4           |2016-03-27 18:07:41|15.94        |1                       |76.5           |76.5              |
|230         |2016-

<a class="anchor" id="2">

# **6. Stop Queries and Spark Session**
</a>

In [14]:
for q in [trips_query, windowed_query, reaction_query]:
    q.stop()

In [15]:
spark.stop()

26/05/30 22:17:01 WARN StateStore: Error running maintenance thread
java.lang.IllegalStateException: SparkEnv not active, cannot do maintenance on StateStores
	at org.apache.spark.sql.execution.streaming.state.StateStore$.doMaintenance(StateStore.scala:632)
	at org.apache.spark.sql.execution.streaming.state.StateStore$.$anonfun$startMaintenanceIfNeeded$1(StateStore.scala:610)
	at org.apache.spark.sql.execution.streaming.state.StateStore$MaintenanceTask$$anon$1.run(StateStore.scala:453)
	at java.base/java.util.concurrent.Executors$RunnableAdapter.call(Executors.java:539)
	at java.base/java.util.concurrent.FutureTask.runAndReset(FutureTask.java:305)
	at java.base/java.util.concurrent.ScheduledThreadPoolExecutor$ScheduledFutureTask.run(ScheduledThreadPoolExecutor.java:305)
	at java.base/java.util.concurrent.ThreadPoolExecutor.runWorker(ThreadPoolExecutor.java:1136)
	at java.base/java.util.concurrent.ThreadPoolExecutor$Worker.run(ThreadPoolExecutor.java:635)
	at java.base/java.lang.Thread.

<a class="anchor" id="1">

# **7. How to run it**

All this commands should be done in ther terminal

**`Install Java (required by Spark)`**

sudo apt-get update
sudo apt-get install -y openjdk-17-jdk-headless

**`Verify Java`**

java -version

**`Install Docker Compose plugin (if not already installed)`**

sudo apt-get install -y docker-compose-plugin

**`Install Python packages`**

pip3 install kafka-python pyspark==3.5.0 ipykernel pendulum yfinance feedparser

**`Start the kafka cluster`**

docker compose -f docker-compose.yml up -d

**`Verify it's running`**

docker ps

**`Create the Kafka topics`**

docker exec -it kafka bash

kafka-topics --create --topic trips \ --bootstrap-server localhost:8098 \ --partitions 1 --replication-factor 1

kafka-topics --create --topic long_trips \ --bootstrap-server localhost:8098 \ --partitions 1 --replication-factor 1

exit

**`Start the producer`**

cd streaming && docker compose up -d && cd ..
nohup python streaming/taxi_producer.py &
tail -f nohup.out

<a class="anchor" id="1">

# **8. Conclusions**
</a>
